# Voxel atlas

Port of `packages/niivue/examples/vox.atlas.html`. Loads the MNI template and AAL atlas with label metadata, then exposes atlas opacity, interpolation, legend, and colorbar controls.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"


def hex_to_rgba(value):
    value = value.lstrip("#")
    return [int(value[i : i + 2], 16) / 255 for i in (0, 2, 4)] + [1.0]


nv = NiiVue(
    background_color=[0.3, 0.3, 0.5, 1.0],
    is_colorbar_visible=True,
    show_render=1,
    backend="webgl2",
)

slice_type = widgets.Dropdown(
    options=[("Axial", 0), ("Coronal", 1), ("Sagittal", 2), ("A+C+S+R", 3), ("Render", 4)],
    value=3,
    description="Slice",
)
opacity = widgets.IntSlider(value=50, min=1, max=100, step=1, description="Opacity")
nearest = widgets.Checkbox(value=True, description="Nearest")
clip_dark = widgets.Checkbox(value=True, description="Clip dark")
legend = widgets.Checkbox(value=False, description="Legend")
colorbar = widgets.Checkbox(value=True, description="Colorbars")
background = widgets.ColorPicker(value="#4d4d80", description="Background")


def update_slice(change):
    nv.slice_type = change["new"]


def update_opacity(change):
    nv.set_volume(1, {"opacity": change["new"] * 0.01})


def update_nearest(change):
    nv.volume_is_nearest_interpolation = change["new"]


def update_clip_dark(change):
    nv.volume_is_alpha_clip_dark = change["new"]


def update_legend(change):
    nv.set_volume(1, {"isLegendVisible": change["new"]})


def update_colorbar(change):
    nv.is_colorbar_visible = change["new"]


def update_background(change):
    nv.background_color = hex_to_rgba(change["new"])


slice_type.observe(update_slice, names="value")
opacity.observe(update_opacity, names="value")
nearest.observe(update_nearest, names="value")
clip_dark.observe(update_clip_dark, names="value")
legend.observe(update_legend, names="value")
colorbar.observe(update_colorbar, names="value")
background.observe(update_background, names="value")

controls = widgets.VBox(
    [
        widgets.HBox([slice_type, opacity, background]),
        widgets.HBox([nearest, clip_dark, legend, colorbar]),
    ]
)

display(controls)
display(nv)

nv.load_volumes(
    [
        {"url": f"{VOLUMES}/mni152.nii.gz", "calMin": 30, "calMax": 80},
        {"url": f"{VOLUMES}/aal.nii.gz"},
    ]
)
nv.set_colormap_label_from_url(1, f"{VOLUMES}/aal.json")
update_nearest({"new": nearest.value})
update_clip_dark({"new": clip_dark.value})
update_opacity({"new": opacity.value})